# PiCo — cell-line drug-response demo

Reproduces slices of **Fig. 3** (zero-shot drug response on held-out cancer types)
and **Fig. 4** (permutation feature importance) from Kirkham et al.
(*Interventionally-guided representation learning for robust and interpretable AI
models in cancer medicine*, 2025) on four headline drugs — AZD6738 (ATR inhibitor),
trametinib (MEK inhibitor), oxaliplatin (platinum chemo), 5-fluorouracil (antimetabolite).

The notebook is focused on modelling: for every (drug × seed × encoder) it
forward-passes the bundled iCoVAE / VAE encoder on the variance-filtered
expression matrix, refits the Stage-2 ElasticNet probe using the saved
hyperparameters, scores on held-out cancer types, and finally computes
permutation feature importance in the same way as the paper. The plotting
code is imported from `src/utils/plot_helpers.py` (lifted verbatim from
`results_analysis/ccl_drug_resp.ipynb`), so the plots look identical to the
published figures — only the underlying data is a 4-drug subset.

Expected runtime on free-tier Colab CPU: **~5 min** (+ ~30 s for the repo clone).

In [ ]:
# --- Setup: Colab pip install + clone; local no-op ---
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    # Clone the repo (with bundled demo assets) and pin a torch version
    # compatible with the current Colab runtime.
    !git clone --depth 1 https://github.com/domkirkham/pico.git
    !pip -q install torch==2.10.0 --index-url https://download.pytorch.org/whl/cpu
    %cd pico

In [ ]:
# --- Imports + path setup ---
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import ElasticNet
from sklearn.preprocessing import StandardScaler

# ElasticNet with l1_ratio=0 prints a "use Ridge instead" warning every fit;
# safe to silence here — the bundled hyperparameters are the paper's, not ours.
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Detect repo root: pico/ (local), pico/demo/ (nbconvert), /content/pico (Colab).
REPO_ROOT = Path().resolve()
if REPO_ROOT.name == "demo":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

import models.pico  # noqa: F401 — registers TabularEncoder for torch.load
from utils.plot_helpers import (
    apply_paper_style, live_permutation_fi,
    plot_perf_comp, plot_perf_pointplot, plot_feature_importance,
)

apply_paper_style(REPO_ROOT / "src" / "fonts")

DEMO_ROOT = REPO_ROOT / "demo" / "assets"
OUT_DIR = REPO_ROOT / "demo" / "outputs"; OUT_DIR.mkdir(parents=True, exist_ok=True)

DEMO_DRUGS = ["AZD6738", "TRAMETINIB", "OXALIPLATIN", "5-FLUOROURACIL"]
SEEDS = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
EXPERIMENT = "h16"
DATASET = "depmap_gdsc"

## 1. Inspect the bundled inputs

The demo reads from `demo/assets/`:

* **Per-(drug × seed) expression CSVs** — 1019 DepMap cell lines × the
  top-1500 variance-filtered genes for each drug-seed combination.
  Variance-filtering depends on the KFold split that Stage 1 used, so each
  seed sees a slightly different gene set.
* **GDSC drug response** — one `LN_IC50` column per demo drug.
* **Cell-line metadata** — `OncotreeLineage` (for 16 held-out cancer types,
  flagged by `h16_held_out`) and the TP53 mutation status.
* **Pretrained encoders** — one iCoVAE + one VAE `encoder_{seed}.pt` per
  drug × seed, plus the Stage-2 ElasticNet hyperparameters
  (`alpha`, `l1_ratio`) in `args_best_s{seed}.txt`.

In [ ]:
# Gene-expression (one matrix per seed, exported by variance filter of Stage 1)
x_sample = pd.read_csv(
    DEMO_ROOT / "data" / f"depmap23q2_expression_AZD6738_s{SEEDS[0]}.csv.gz",
    index_col=0,
)
print(f"Expression matrix shape (AZD6738 seed {SEEDS[0]}):", x_sample.shape)
print(x_sample.iloc[:3, :5])

# GDSC LN(IC50) for the four demo drugs
gdsc = pd.read_csv(DEMO_ROOT / "data" / "gdsc2_demo_drugs_ln_ic50.csv", index_col=0)
print(f"\nGDSC drug-response shape: {gdsc.shape}")
print(gdsc.head(3))

# Cell-line metadata (OncotreeLineage + h16 flag)
model = pd.read_csv(DEMO_ROOT / "data" / "depmap23q2_model.csv", index_col=0)
print(f"\nModel metadata: {len(model)} lines; h16 held-out: "
      f"{int(model['h16_held_out'].sum())}")
print(model.head(3))

## 2. Load one encoder — architecture + single forward pass

Quick sanity check that the bundled encoder files load correctly and produce
a 32-dim latent representation (the value of `z_dim` in the Stage-1
hyperparameter file). Subsequent cells will do this at scale.

In [ ]:
drug, seed = "AZD6738", SEEDS[0]
enc_path = (DEMO_ROOT / "data" / "outputs" / DATASET / drug / EXPERIMENT
            / "icovae" / f"encoder_{seed}.pt")
encoder = torch.load(enc_path, map_location="cpu", weights_only=False)
encoder.eval()

stage1_cfg = json.loads(
    (DEMO_ROOT / "data" / "outputs" / DATASET / drug / EXPERIMENT
     / "icovae" / "args_best.txt").read_text()
)
print(f"Stage-1 z_dim = {stage1_cfg['z_dim']}; constraints = "
      f"{stage1_cfg['curr_constraints']}")
print("\nEncoder module:")
print(encoder)

x = torch.as_tensor(x_sample.iloc[:1].to_numpy(), dtype=torch.float32)
with torch.no_grad():
    loc, scale = encoder(x)
print(f"\nOne-sample forward pass:\n  input   {tuple(x.shape)}"
      f"\n  loc     {tuple(loc.shape)}\n  scale   {tuple(scale.shape)}")

## 3. Live encoding + Stage-2 probe fit

For every (drug × encoder × seed) we:

1. Load the variance-filtered expression matrix bundled for that drug-seed.
2. Forward-pass the matching pretrained encoder to obtain the 32-dim
   latent `z`.
3. Identify training / test samples using the cell-line metadata: any line
   in one of the 16 held-out cancer types is a test sample.
4. Fit a `StandardScaler` on the training `z` (the same scaler that
   `PiCoSK.fit_regressor` fits in `src/models/pico.py`).
5. Fit an ElasticNet with the saved `alpha` / `l1_ratio` hyperparameters
   on scaled training `z` and GDSC LN(IC50).
6. Score on the held-out test samples → Spearman ρ, Pearson ρ, RMSE.

Cache the fitted coefficients + scaled test matrix + test labels for the
live permutation-FI step below.

In [ ]:
perf_rows = []
fit_cache = {}  # (drug, enc_type, seed) -> dict(coef, intercept, z_test_s, y_test)

for drug in DEMO_DRUGS:
    for enc_type in ("icovae", "vae"):
        for seed in SEEDS:
            x_df = pd.read_csv(
                DEMO_ROOT / "data"
                / f"depmap23q2_expression_{drug}_s{seed}.csv.gz",
                index_col=0,
            )

            enc = torch.load(
                DEMO_ROOT / "data" / "outputs" / DATASET / drug / EXPERIMENT
                / enc_type / f"encoder_{seed}.pt",
                map_location="cpu", weights_only=False,
            )
            enc.eval()
            with torch.no_grad():
                z_loc, _ = enc(torch.as_tensor(x_df.to_numpy(), dtype=torch.float32))
            z_all = z_loc.numpy()

            cfg = json.loads(
                (DEMO_ROOT / "data" / "outputs" / DATASET / drug / EXPERIMENT
                 / "pico" / f"ElasticNet_{enc_type}"
                 / f"args_best_s{seed}.txt").read_text()
            )
            test_set = set(cfg["test_samples"])

            y = gdsc[drug].reindex(x_df.index)
            train_mask = np.array([(s not in test_set) and (not pd.isna(y.loc[s]))
                                   for s in x_df.index])
            test_mask = np.array([s in test_set for s in x_df.index])

            scaler = StandardScaler().fit(z_all[train_mask])
            z_train_s = scaler.transform(z_all[train_mask])
            z_test_s = scaler.transform(z_all[test_mask])

            reg = ElasticNet(
                alpha=cfg["alpha"], l1_ratio=cfg["l1_ratio"],
                max_iter=10000, random_state=seed,
            ).fit(z_train_s, y[train_mask].to_numpy())

            pred = reg.predict(z_test_s)
            y_test = y[test_mask].to_numpy()
            nas = np.isnan(y_test) | np.isnan(pred)
            perf_rows.append({
                "target": drug, "fe": enc_type, "model": "ElasticNet",
                "seed": seed,
                "spearman_r": spearmanr(pred[~nas], y_test[~nas])[0],
                "pearson_r": pearsonr(pred[~nas], y_test[~nas])[0],
                "rmse": float(np.sqrt(np.mean((pred[~nas] - y_test[~nas]) ** 2))),
            })
            fit_cache[(drug, enc_type, seed)] = {
                "coef": reg.coef_, "intercept": reg.intercept_,
                "z_test_s": z_test_s, "y_test": y_test,
            }

perf_df = pd.DataFrame(perf_rows)
print(perf_df.groupby(["target", "fe"])[["spearman_r", "pearson_r", "rmse"]]
      .mean().round(3))

## 4. Figure 3 — aggregate performance

Each (feature extractor × regression model) combination collapses to its
mean across seeds before plotting. Call imported from `plot_helpers.py`
(`plot_perf_comp` is a near-verbatim extract of
`results_analysis/ccl_drug_resp.ipynb` cell 39).

In [ ]:
plot_perf_comp(perf_df, plot_metric="spearman_r",
               save_to=OUT_DIR / "fig3_spearman_r")
plt.show()

### Per-drug breakdown (`plot_perf_pointplot`)

The per-drug pointplot view from cell 42 of the paper notebook — every
(FE × regression model) combination shown as a separate point per drug,
with ±1 s.d. error bars across seeds. Drugs are ordered by mean iCoVAE
Spearman ρ.

In [ ]:
plot_perf_pointplot(perf_df, plot_metric="spearman_r",
                    save_to=OUT_DIR / "fig3_pointplot_spearman_r")
plt.show()

## 5. Figure 4 — permutation feature importance

For each latent dimension `z_i` we shuffle its values across the held-out
test set 100 times and record the Δ in Spearman ρ. This is the exact
procedure used in `results_analysis/ccl_drug_resp.ipynb` (the production
version is in `utils.comp_utils.calculate_feat_imps`, which reads cached
prediction files; here we do it live from the fitted regressor coefficients
and the already-scaled test matrix).

We show Fig. 4b (AZD6738) etc. for all four demo drugs; the dominant
`z_{constraint}` dimension should match the paper (e.g. `z_9`, which is
the `CCND1` constraint for AZD6738).

In [ ]:
for drug in DEMO_DRUGS:
    # iCoVAE labels its first len(constraints) latent dims with the
    # constraint gene names (the rest are unconstrained).  Mirrors the
    # `rep_renamer` mapping in src/utils/comp_utils.py so the FI bar plot
    # shows e.g. z_BRAF instead of z_0 for trametinib.
    constraints = json.loads(
        (DEMO_ROOT / "data" / "outputs" / DATASET / drug / EXPERIMENT
         / "icovae" / "args_best.txt").read_text()
    )["curr_constraints"]
    fi_frames = []
    for seed in SEEDS:
        fit = fit_cache[(drug, "icovae", seed)]
        n_z = fit["z_test_s"].shape[1]
        feature_names = (list(constraints)
                         + [str(i) for i in range(len(constraints), n_z)])
        feature_names = [f"z_{name}" for name in feature_names]
        fi_frames.append(live_permutation_fi(
            fit["z_test_s"], fit["y_test"],
            feature_names=feature_names,
            coeffs=fit["coef"], intercept=fit["intercept"],
            seed=seed, n_iter=100, classification=False,
        ))
    pi_df = pd.concat(fi_frames, axis=0, ignore_index=True)
    plot_feature_importance(
        pi_df, target=drug, fe="icovae", model="ElasticNet", metric="s",
        save_to=OUT_DIR / f"fig4_{drug}",
    )
    plt.show()

## 6. Save aggregated results

In [ ]:
perf_df.to_csv(OUT_DIR / "ccl_perf_df.csv", index=False)
summary = (perf_df.groupby(["target", "fe"])["spearman_r"]
           .agg(["mean", "std", "count"]).round(3).reset_index())
summary.to_csv(OUT_DIR / "ccl_spearman_summary.csv", index=False)
summary